In [54]:
import pandas as pd 
import plotly.express as px 
import warnings
import plotly.io as pio
import plotly.graph_objects as go
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
import joblib
import os

pio.templates.default = "plotly_white" 
warnings.filterwarnings('ignore')

In [2]:
df = pd.read_csv('../data/AIML Dataset.csv')
df.sample(5)

,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
279590,15,CASH_OUT,112158.64,C1898063141,0.00,0.00,C1036804658,829888.47,942047.11,0,0
1358210,138,CASH_OUT,283672.38,C1663224226,63542.00,0.00,C97858915,247797.84,531470.22,0,0
2290972,187,CASH_IN,182031.02,C94787636,52807.00,234838.02,C1653558528,50590.69,0.00,0,0
2012488,180,CASH_IN,64439.63,C1013892711,1939835.72,2004275.35,C1869637748,467885.23,403445.61,0,0
6208573,587,CASH_IN,158053.24,C1171779981,11636.00,169689.24,C210535094,0.00,0.00,0,0


In [3]:
# ----------------------------
# ELIMINAMOS LAS COLUMNAS INNECESARIAS
# ----------------------------

df_model = df.drop(['step', 'nameOrig', 'nameDest', 'isFlaggedFraud'], axis = 1)

In [4]:
# ----------------------------
# DIVIDIMOS EL DATASET PARA TRAIN Y TEST
# ----------------------------

y = df_model['isFraud']
X = df_model.drop('isFraud', axis = 1)

# ----------------------------
# SELECCIONAMOS COLUMNAS CATEGÓRICAS Y NUMÉRICAS
# ----------------------------

categorical_columns = X.select_dtypes(include = ['object', 'category']).columns.tolist()
numerical_columns = X.select_dtypes(include = 'number').columns.tolist()

print('Las Columnas de Tipo Categórica son: ', categorical_columns)
print('Las Columnas de Tipo Numérica son (excluyendo si es Fraude): ', numerical_columns)

# ----------------------------
# REALIZAMOS EL TRAIN Y TEST
# ----------------------------
# Se Usa Stratify Para Asegurar Misma Proporción de Fraude en Train y Test
# Se implementa test size de 0.3 para tener una evaluación más robusta al ser un dataset grande y desbalanceado, de alto riesgo (fraude)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.3, stratify = y)

Las Columnas de Tipo Categórica son:  ['type']
Las Columnas de Tipo Numérica son (excluyendo si es Fraude):  ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest']


In [ ]:
# ----------------------------
# INICIAMOS EL PREPROCESAMIENTO
# ----------------------------

preprocessor = ColumnTransformer(
    transformers= [
        ('num', StandardScaler(), numerical_columns),
        ('cat', OneHotEncoder(drop='first'), categorical_columns)
    ],
    remainder= 'drop'
)

In [ ]:
# ----------------------------
# DECLARAMOS TODOS LOS MODELOS A PROBAR
# ----------------------------

models = {
    "LogReg": LogisticRegression(class_weight= 'balanced', max_iter=1000),
    "RandomForest": RandomForestClassifier(n_estimators=100, class_weight="balanced", random_state=42, n_jobs=-1),
    "GradientBoosting": GradientBoostingClassifier(),
    "XGBoost": XGBClassifier(
        n_estimators=200,
        max_depth=6,
        learning_rate=0.1,
        eval_metric="logloss",
        random_state=42,
        scale_pos_weight=10
    )
}

In [ ]:
results = []
roc_curves = {}

for name, model in models.items():

    # ----------------------------
    # INICIAMOS EL PIPELINE PARA EL ENTRENAMIENTO DE CADA MODELO
    # ----------------------------
    pipeline_model = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    # ----------------------------
    # REALIZAMOS EL ENTRENAMIENTO DE CADA MODELO
    # ----------------------------
    pipeline_model.fit(X_train, y_train)

    # ----------------------------
    # REALIZAMOS LAS PREDICCIONES DE CADA MODELO
    # ----------------------------
    y_pred = pipeline_model.predict(X_test)

    # ----------------------------
    # GUARDAMOS LOS RESULTADOS DE CADA MODELO
    # ----------------------------
    if hasattr(pipeline_model, "predict_proba"):
        y_prob = pipeline_model.predict_proba(X_test)[:, 1]
    else:
        y_prob = y_pred

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob)
    })

    roc_curves[name] = y_prob

In [ ]:
# ----------------------------
# CONVERTIMOS LOS RESULTADOS DE CADA MODELO EN UN DATAFRAME
# ----------------------------

results_df = pd.DataFrame(results).sort_values(by="Recall", ascending=False)
results_df

,Model,Accuracy,Precision,Recall,F1,ROC-AUC
3,XGBoost,0.999359,0.681486,0.945617,0.792113,0.999068
0,LogReg,0.947254,0.022500,0.939123,0.043947,0.988997
1,RandomForest,0.999677,0.897889,0.845779,0.871055,0.996346
2,GradientBoosting,0.999026,0.703978,0.423701,0.529009,0.711659


In [43]:
# ----------------------------
# MOSTRAMOS LOS RESULTADOS DE CADA MODELO EN UN GRÁFICO INTERACTIVO
# ----------------------------
metrics = ["Precision", "Recall", "F1", "ROC-AUC"]

results_plot = (
    results_df
    .melt(
        id_vars="Model",
        value_vars=metrics,
        var_name="Métrica",
        value_name="Valor"
    )
)

fig = px.bar(
    results_plot,
    x="Model",
    y="Valor",
    color="Métrica",
    barmode="group",
    text_auto=".3f",
    color_discrete_sequence=[
        "#584ac3",  
        "#eb8e3e",  
        "#2ca070",  
        "#663871"   
    ],
    title="<b>Comparación de Modelos para la Detección de Fraude</b>"
)

fig.update_layout(
    template="plotly_white",
    title_x=0.5,
    yaxis_title="Puntuación",
    xaxis_title="Modelo",
    legend_title="Métrica",
    yaxis=dict(range=[0, 1]),
    font=dict(size=14)
)

fig.show()

fig.write_image("../images/resultados_modelos.png")

In [50]:
# ----------------------------
# MOSTRAMOS LA CURVA ROC DE CADA MODELO
# ----------------------------

colors = {
    "LogReg": "#584ac3",
    "RandomForest": "#eb8e3e",
    "GradientBoosting": "#2ca070",
    "XGBoost": "#663871"
}

fig = go.Figure()

for name, y_prob in roc_curves.items():

    fpr, tpr, _ = roc_curve(y_test, y_prob)
    auc = roc_auc_score(y_test, y_prob)

    fig.add_trace(
        go.Scatter(
            x=fpr,
            y=tpr,
            mode="lines",
            name=f"{name} (AUC = {auc:.3f})",
            line=dict(
                color=colors[name],
                width=3
            ),
            hovertemplate=(
                "<b>%{fullData.name}</b><br>"
                "FPR: %{x:.3f}<br>"
                "TPR: %{y:.3f}<extra></extra>"
            )
        )
    )

# Línea diagonal (clasificador aleatorio)
fig.add_trace(
    go.Scatter(
        x=[0, 1],
        y=[0, 1],
        mode="lines",
        name="Clasificador aleatorio",
        line=dict(
            color="gray",
            dash="dash",
            width=2
        ),
        hoverinfo="skip"
    )
)

fig.update_layout(
    template="plotly_white",
    title="<b>Comparación de Curvas ROC</b>",
    title_x=0.5,
    width=900,
    height=650,
    xaxis=dict(
        title="<b>False Positive Rate (FPR)</b>",
        range=[0, 1],
        showgrid=True,
        gridcolor="rgba(180,180,180,0.25)"
    ),
    yaxis=dict(
        title="<b>True Positive Rate (TPR)</b>",
        range=[0, 1.02],
        showgrid=True,
        gridcolor="rgba(180,180,180,0.25)"
    ),
    legend=dict(
        title="<b>Modelos</b>",
        y=0.02,
        x=0.98,
        xanchor="right",
        yanchor="bottom",
        bgcolor="rgba(255,255,255,0.7)"
    ),
    font=dict(size=14)
)

fig.show()

fig.write_image("../images/resultados_modelos_roc_curve.png")


In [56]:
# ----------------------------
# GUARDAR EL MEJOR MODELO
# ----------------------------
# Selecconamos el mejor modelo según 
best_model_name = results_df.sort_values(by="Recall", ascending=False).iloc[0]["Model"]
print("Mejor modelo:", best_model_name)

# Lo volvemos a entrenar
best_model = models[best_model_name]
final_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", best_model)
])

final_pipeline.fit(X_train, y_train)

os.makedirs("models", exist_ok=True) # Nos aseguramos que la carpeta models existe
joblib.dump(final_pipeline, f"../models/best_model_{best_model_name}.joblib")

Mejor modelo: XGBoost


['../models/best_model_XGBoost.joblib']